# Clinician Adoption Personas: a multi-agent pipeline

This notebook implements the adoption-persona framework (Alrefaei et al.) with groups of Claude agents. **The researcher defines the adoption attributes**; the agents then extract, rate, cluster and build personas from them.

| # | Agent group | What it does |
|---|---|---|
| 1 | **Familiarization & speaker identification** | One *Reader* per transcript works out who is the interviewer and who is the participant, and assigns the participant's **role** (physician / nurse). A *Lead* checks the roles across the dataset. |
| 2 | **Thematic analysis** | One *Coder* per transcript, then a *Theme Lead* and a *Theme Reviewer*. Themes are flagged as shared, role-specific or divergent between physicians and nurses. |
| 3 | **Statement extraction & attribute rating** | An *Extractor* pulls every participant statement that fits a construct definition. Two independent *Raters* score each statement 1–5, reading its meaning and sentiment. An *Adjudicator* resolves disagreements. Ratings are averaged into participant-level attribute scores. |
| 4 | **Cluster analysis** | Hierarchical clustering **separately for each role** (z-scores, Euclidean distance, complete linkage). A *Cluster Analyst* checks candidate cut points for qualitative coherence; **you confirm the cut**. A *Profile Matcher* merges physician and nurse clusters with the same adoption pattern into shared profiles. |
| 5 | **Persona cards** | One *Persona Designer* per adoption profile writes the persona, role-specific needs, design requirements and implementation strategies. Quotes are checked against the transcripts. |

Optional extras from the paper: **RoBERTa sentiment features** (CardiffNLP) and a **post hoc correlation** of SUS / perceived ease of use with the attributes.

### How to run in Google Colab
1. Click the 🔑 **Secrets** icon. Add `ANTHROPIC_API_KEY` and turn on **Notebook access**.
2. Review the **researcher input** cells: attributes (Step 3) and study settings (Step 2).
3. **Runtime → Run all.** The run pauses at Step 9 so you can confirm the cluster cut points.

The built-in sample transcripts are short and fictional. They only demonstrate the flow.

## Step 1: Install

In [ ]:
!pip install -q anthropic python-docx scikit-learn transformers

## Step 2: Study settings and connection to Claude

In [ ]:
import anthropic, json, os, re
from concurrent.futures import ThreadPoolExecutor

try:
    from google.colab import userdata
    API_KEY = userdata.get("ANTHROPIC_API_KEY")
except ImportError:
    API_KEY = os.environ["ANTHROPIC_API_KEY"]
client = anthropic.Anthropic(api_key=API_KEY)

MODEL = "claude-opus-5-5"
MAX_PARALLEL = 4                 # agents running at the same time; lower it if you hit rate limits
OUTPUT_DIR = "persona_outputs"
os.makedirs(OUTPUT_DIR, exist_ok=True)

# ---------------- RESEARCHER INPUT ----------------
RESEARCH_CONTEXT = """
System: an AI-enabled clinical decision support system (CDSS) for chronic pain assessment.
It estimates aspects of the patient's pain experience from eye-movement measures and presents
them in a clinical dashboard. Data: think-aloud sessions while clinicians used the dashboard
on a representative chronic pain case, semistructured interviews (informed by Innovation
Diffusion Theory and the Technology Acceptance Model), and clinicians' explanations of their
SUS and perceived ease-of-use ratings.
Goal: develop clinician adoption personas to inform design and implementation.
"""
USER_GROUPS = ["Physician", "Nurse"]     # the professional roles clustered separately
USE_ROBERTA_SENTIMENT = True             # add RoBERTa sentiment scores as separate clustering features
# ---------------------------------------------------


def run_agent(role: str, task: str, output_model=None, effort: str = "medium"):
    """Run one agent. role = its system prompt. With output_model (a Pydantic class),
    the answer comes back as a validated object instead of text."""
    kwargs = dict(
        model=MODEL, max_tokens=64000, system=role,
        messages=[{"role": "user", "content": task}],
        output_config={"effort": effort},
        betas=["server-side-fallback-2026-07-01"],
        fallbacks="default",                     # if Claude declines, retry on a fallback model
    )
    if output_model is not None:
        kwargs["output_format"] = output_model
    with client.beta.messages.stream(**kwargs) as stream:
        msg = stream.get_final_message()
    if msg.stop_reason == "refusal":
        raise RuntimeError(f"Request declined: {msg.stop_details}")
    if msg.stop_reason == "max_tokens":
        raise RuntimeError("Answer was cut off (max_tokens).")
    if output_model is not None:
        return msg.parsed_output
    return "".join(b.text for b in msg.content if b.type == "text")

def parallel(fn, items):
    with ThreadPoolExecutor(max_workers=MAX_PARALLEL) as pool:
        return list(pool.map(fn, items))

def save_json(name, obj):
    with open(os.path.join(OUTPUT_DIR, name), "w", encoding="utf-8") as f:
        json.dump(obj, f, ensure_ascii=False, indent=2, default=str)

print(run_agent("You are a helpful assistant.", "Reply with the single word: ready", effort="low"))

## Step 3: Researcher input, the adoption attributes

**This step is the researcher's job, not the agents'.** Define each construct, its theoretical source, and what a low (1) and high (5) rating means. The agents extract statements and rate them only against these definitions. Edit the wording freely, and add or remove attributes. Everything downstream adapts.

In [ ]:
ATTRIBUTES = {
    "fit_with_workflow": {
        "name": "Fit with clinical workflow",
        "source": "Innovation Diffusion Theory (compatibility); sociotechnical perspective (Sittig & Singh, 2010)",
        "definition": "The degree to which the clinician perceives the system as compatible with their existing "
                      "clinical workflow, routines, time constraints and professional responsibilities.",
        "low": "The system would disrupt the workflow, add steps or time, or not fit their responsibilities.",
        "high": "The system fits naturally into existing routines and responsibilities without extra burden.",
    },
    "usability": {
        "name": "Usability",
        "source": "System Usability Scale (Brooke, 1996)",
        "definition": "The clinician's perception of how well the dashboard supports effective, efficient and "
                      "satisfying use: layout, clarity, organization and presentation of information.",
        "low": "The interface is confusing, cluttered, inconsistent or hard to interpret.",
        "high": "The interface is clear, well organized and pleasant to use.",
    },
    "trust_in_ai": {
        "name": "Trust in AI",
        "source": "Trust in AI-based CDSS (Schwartz et al., 2022; Tun et al., 2025)",
        "definition": "The clinician's confidence in the credibility, accuracy, reliability and transparency of "
                      "the AI-generated, eye-movement-based pain information.",
        "low": "Doubts the validity or accuracy of the AI output; would not rely on it.",
        "high": "Considers the AI output credible and reliable enough to inform clinical decisions.",
    },
    "relative_advantage": {
        "name": "Relative advantage",
        "source": "Innovation Diffusion Theory (Rogers, 2003)",
        "definition": "The degree to which the clinician perceives the system as better than current practice "
                      "for chronic pain assessment (e.g., adds objective information, improves decisions or care).",
        "low": "Sees no added value over current practice and self-report measures.",
        "high": "Sees clear, substantial clinical value over current practice.",
    },
    "ease_of_use": {
        "name": "Ease of use",
        "source": "Technology Acceptance Model (Davis, 1989)",
        "definition": "The degree to which the clinician believes using the system would be free of effort: "
                      "easy to learn, operate and interpret.",
        "low": "Using or learning the system would require substantial effort or training.",
        "high": "The system would be effortless to learn and use.",
    },
    "intention_to_use": {
        "name": "Intention to use",
        "source": "Technology Acceptance Model (Davis, 1989)",
        "definition": "The clinician's stated willingness or plan to use the system in their clinical practice.",
        "low": "Would not use the system.",
        "high": "Clearly intends to use the system in practice.",
    },
}
ATTRIBUTE_KEYS = list(ATTRIBUTES)
SHORT = {k: v["name"] for k, v in ATTRIBUTES.items()}

def attribute_brief(keys=ATTRIBUTE_KEYS):
    return "\n\n".join(
        f"[{k}] {ATTRIBUTES[k]['name']} ({ATTRIBUTES[k]['source']})\n"
        f"Definition: {ATTRIBUTES[k]['definition']}\n1 (low): {ATTRIBUTES[k]['low']}\n5 (high): {ATTRIBUTES[k]['high']}"
        for k in keys)
print(f"{len(ATTRIBUTES)} attributes:", ", ".join(SHORT.values()))

## Step 4: Load the transcripts

Use one file per participant (`.txt` or `.docx`). Speaker labels can be anything. Group 1 works out who is who.

In [ ]:
USE_SAMPLE_DATA = True

SAMPLE_TRANSCRIPTS = {
"C01.txt": """Interviewer: Please go through the dashboard as you would in a consultation.
Speaker 2: OK, so this is the pain estimate from the eye movements. I like that it sits next to the medication list and the pain history. That's what I'd look at anyway.
Interviewer: What do you think of the information?
Speaker 2: For chronic pain this is something we really lack, an objective measure. If it's validated it would help me a lot in deciding whether a treatment is working.
Interviewer: Would you trust it?
Speaker 2: I'd want to know how the estimate is calculated, but if there's an explanation I'd trust it alongside my own judgment.
Interviewer: Would you use it?
Speaker 2: Yes, definitely, I'd use it for my follow-up patients. Adding the eye-tracking step to the visit might take time though, who would do it?
Interviewer: You gave the system a high usability score. Why?
Speaker 2: It's clean, I found everything quickly, very easy to use.""",

"C02.txt": """Speaker A: I'm an anesthesiologist, I've done pain medicine for twenty years.
Speaker B: Thanks. Please think aloud as you use the dashboard.
Speaker A: Honestly I don't see what this adds. Pain is what the patient tells me. An eye-movement number won't change my treatment.
Speaker B: How about trusting the estimate?
Speaker A: I don't trust it. It's a black box, and I haven't seen the evidence.
Speaker B: Would you use it?
Speaker A: No. Not unless there are large trials showing benefit. The screen is easy enough, that's not the issue, it's that it slows my clinic down.""",

"C03.txt": """Interviewer: Can you walk me through the case using the dashboard?
Dr. Reem: Sure. Wow, the longitudinal trend is great, I can see the pain trajectory across visits. I'd love opioid alerts here too.
Interviewer: How does it compare to what you do now?
Dr. Reem: Much better than a 0 to 10 scale alone. It gives me something objective to discuss with the patient.
Interviewer: Trust?
Dr. Reem: I trust it as one input. I'd use it tomorrow if I could.
Interviewer: Fit with your workflow?
Dr. Reem: It would need to be in the EHR, otherwise it's another login. But I'd make it work.
Interviewer: Ease of use?
Dr. Reem: Very intuitive, I didn't need any help.""",

"C04.txt": """Interviewer: Please think aloud as you look at the dashboard.
Speaker 2: There's a lot here. I'm not sure what this AI percentage means. I'd prefer familiar terms.
Interviewer: Would it help your practice?
Speaker 2: It could, maybe, for patients who struggle to describe pain. But I'd need to see it work in our clinic first.
Interviewer: Do you trust the estimate?
Speaker 2: Somewhat. I'd want to compare it with what the patient says for a while.
Interviewer: Would you use it?
Speaker 2: Possibly, if it doesn't add work. Our visits are fifteen minutes. A simpler version with just the essentials would help.""",

"C05.txt": """Interviewer: Tell me what you see as you use the dashboard.
Speaker 2: The layout is clear, the pain estimate is easy to find. I'd want the vitals and history next to it to interpret it.
Interviewer: How useful is it compared to now?
Speaker 2: Useful for monitoring treatment response, it adds a piece we don't have.
Interviewer: Trust in the information?
Speaker 2: I'd trust it if it explained why it gives that estimate. Right now it doesn't tell me why.
Interviewer: Would you use it?
Speaker 2: Yes, I think I would, especially if it fits into the EHR and doesn't add steps.""",

"N01.txt": """Interviewer: You're a registered nurse in a pain clinic. Please think aloud using the dashboard.
Speaker 2: OK. So I'd be the one collecting the eye-movement data at intake? That's a new task for us, and intake is already busy.
Interviewer: What do you think of the information?
Speaker 2: It would help me flag patients for the physician. That's valuable, it supports what patients tell us.
Interviewer: Trust?
Speaker 2: I'm not sure yet. I'd need training on what the numbers mean.
Interviewer: Would you use it?
Speaker 2: If the clinic adopts it, yes, but I'd like a simpler nurse view for documentation.""",

"N02.txt": """Speaker 1: Can you introduce yourself?
Speaker 2: I'm an oncology nurse, about eight years.
Speaker 1: Please use the dashboard and think aloud.
Speaker 2: This is exciting! Patients often under-report pain, so having an objective measure is a big deal. I trust it as long as it's validated.
Speaker 1: Ease of use?
Speaker 2: Very easy, the screen is clear and I'd learn it in minutes.
Speaker 1: Would you use it?
Speaker 2: Absolutely, I'd use it with every chronic pain patient and share it with the doctors.""",

"N03.txt": """Interviewer: Please walk me through the dashboard.
Speaker 2: Hmm, there are many sections. I'm a community health nurse, I don't usually see this kind of data.
Interviewer: Is it useful compared to now?
Speaker 2: It could add something, but I'm cautious. I'd want to see it working before relying on it.
Interviewer: Trust?
Speaker 2: Not fully. How accurate is the eye tracking?
Interviewer: Would you use it?
Speaker 2: Maybe, if it fits our home visits and doesn't need a lot of training. A short version would be better for us.""",
}

def read_file(name, data):
    if name.lower().endswith(".docx"):
        import docx, io
        return "\n".join(p.text for p in docx.Document(io.BytesIO(data)).paragraphs)
    return data.decode("utf-8", errors="ignore")

if USE_SAMPLE_DATA:
    raw_files = SAMPLE_TRANSCRIPTS
else:
    from google.colab import files
    raw_files = {n: read_file(n, d) for n, d in files.upload().items()}

transcripts = [{"participant_id": os.path.splitext(name)[0], "file": name, "raw_text": text}
               for name, text in sorted(raw_files.items())]
for t in transcripts:
    print(t["participant_id"], f"{len(t['raw_text']):,} chars")

## Step 5: Output schemas

In [ ]:
from pydantic import BaseModel, Field
from typing import Literal, Optional

AttributeKey = Literal[tuple(ATTRIBUTE_KEYS)]

class SpeakerRole(BaseModel):
    speaker_label: str = Field(description="Label exactly as written in the transcript, e.g. 'Speaker 2', 'Dr. Reem'")
    role: Literal["interviewer", "participant", "other"]
    reasoning: str

class Familiarization(BaseModel):
    speakers: list[SpeakerRole]
    user_group: str = Field(description="The participant's professional role, chosen from the given user groups")
    user_group_evidence: str
    profile: str = Field(description="Specialty, experience and other details the participant states; 'not stated' otherwise")
    summary: str
    initial_observations: list[str]

class RoleAssignment(BaseModel):
    participant_id: str
    user_group: str

class DatasetOverview(BaseModel):
    overview: str
    assignments: list[RoleAssignment]
    data_quality_notes: list[str]

class Code(BaseModel):
    code: str
    description: str
    quote: str

class CodingResult(BaseModel):
    codes: list[Code]

class Theme(BaseModel):
    theme: str
    definition: str
    sub_themes: list[str]
    participant_ids: list[str]
    groups_present: list[str]
    group_flag: Literal["shared across groups", "group-specific", "divergent across groups"]
    group_differences: str
    related_attributes: list[str]
    key_quotes: list[str]

class ThematicMap(BaseModel):
    themes: list[Theme]
    cross_group_summary: str

class ExtractedStatement(BaseModel):
    attribute: AttributeKey
    statement: str = Field(description="The participant's words copied verbatim")
    why_it_fits: str

class Extraction(BaseModel):
    statements: list[ExtractedStatement]
    attributes_without_evidence: list[str]

class StatementRating(BaseModel):
    statement_id: int
    rating: int = Field(description="1 (low level of the attribute) to 5 (high level)")
    rationale: str

class RatingSheet(BaseModel):
    ratings: list[StatementRating]

class CutRecommendation(BaseModel):
    recommended_k: int
    coherence_by_k: list[str] = Field(description="For each candidate k, whether each cluster is qualitatively coherent")
    rationale: str

class Profile(BaseModel):
    profile_name: str = Field(description="Short adoption profile name, e.g. 'Cautious Evaluator'")
    clusters: list[str] = Field(description="Cluster IDs merged into this profile, e.g. 'Physician-2'")
    adoption_pattern: str
    role_differences: str

class ProfileMatching(BaseModel):
    profiles: list[Profile]
    rationale: str

class RoleNeeds(BaseModel):
    role: str
    needs: list[str]

class PersonaCard(BaseModel):
    persona_name: str
    profile_name: str
    tagline_quote: str = Field(description="A verbatim participant quote")
    description: str = Field(description="3-4 sentences describing how this profile evaluates and adopts the system")
    goals: list[str]
    adoption_drivers: list[str]
    adoption_concerns: list[str]
    conditions_for_use: list[str]
    role_specific_needs: list[RoleNeeds]
    design_requirements: list[str]
    implementation_strategies: list[str]
    supporting_quotes: list[str] = Field(description="2-4 verbatim participant quotes")

## Step 6: Agent group 1, familiarization and speaker identification

In [ ]:
READER = f"""You are a qualitative researcher in the familiarization phase of thematic analysis.
Read the whole transcript (think-aloud, interview and questionnaire explanations).
1. Identify every speaker label: who is the interviewer (asks questions, gives instructions) and who is
   the participant (describes their own experience).
2. Determine the participant's professional role, choosing from: {USER_GROUPS}. Quote your evidence.
   If the role is not stated, infer cautiously and say so.
3. Summarize the session and note initial observations. Never invent details.
Research context: {RESEARCH_CONTEXT}"""

FAMILIARIZATION_LEAD = f"""You lead the familiarization phase. Review all readers' notes, confirm every
participant's role (one of {USER_GROUPS}), and note data-quality issues (unclear speakers, unclear role,
very short sessions)."""

def relabel(text, speakers, pid):
    """Rewrite speaker labels as INTERVIEWER / PARTICIPANT so later agents never mix them up."""
    for s in sorted(speakers, key=lambda s: -len(s.speaker_label)):
        new = {"interviewer": "INTERVIEWER", "participant": f"PARTICIPANT {pid}"}.get(s.role, "OTHER")
        text = re.sub(rf"(?im)^(\s*(?:\[[^\]]*\]\s*)?){re.escape(s.speaker_label)}(\s*(?:\([^)]*\))?\s*:)",
                      rf"\g<1>{new}\g<2>", text)
    return text

def read_transcript(t):
    r = run_agent(READER, f"Participant ID: {t['participant_id']}\n\nTranscript:\n{t['raw_text']}", Familiarization)
    print(f"  Reader finished {t['participant_id']}")
    return r

print("Group 1: readers are working...")
for t, fam in zip(transcripts, parallel(read_transcript, transcripts)):
    t["familiarization"] = fam
    t["clean_text"] = relabel(t["raw_text"], fam.speakers, t["participant_id"])

overview = run_agent(FAMILIARIZATION_LEAD, "\n\n".join(
    f"## {t['participant_id']}\n{t['familiarization'].model_dump_json(indent=1)}" for t in transcripts), DatasetOverview)
confirmed = {a.participant_id: a.user_group for a in overview.assignments}
for t in transcripts:
    t["user_group"] = confirmed.get(t["participant_id"], t["familiarization"].user_group)

by_pid = {t["participant_id"]: t for t in transcripts}
for t in transcripts:
    roles = ", ".join(f"{s.speaker_label} = {s.role}" for s in t["familiarization"].speakers)
    print(f"{t['participant_id']:>5} [{t['user_group']}]  {roles}")
print("\nParticipants per role:", {g: sum(t["user_group"] == g for t in transcripts) for g in USER_GROUPS})
for note in overview.data_quality_notes:
    print("  ⚠", note)
save_json("1_familiarization.json", {"overview": overview.model_dump(), "participants": [
    {"participant_id": t["participant_id"], "user_group": t["user_group"], **t["familiarization"].model_dump(),
     "clean_transcript": t["clean_text"]} for t in transcripts]})

## Step 7: Agent group 2, thematic analysis with role differences flagged

In [ ]:
CODER = f"""You are a qualitative coder (Braun & Clarke, phase 2). Code ONLY the PARTICIPANT's words.
Create concise codes about the clinician's experience with the system, their evaluation of it and their
adoption perceptions. Every code needs a short verbatim quote. Research context: {RESEARCH_CONTEXT}"""

THEME_LEAD = f"""You are a senior qualitative researcher (Braun & Clarke, phases 3-5). Organize all codes
into themes. For each theme record the participants and roles it appears in, and FLAG it:
"shared across groups", "group-specific" (only or mainly one role) or "divergent across groups"
(roles talk about it in contrasting ways). Also list which adoption attributes it relates to:
{list(SHORT.values())}. Research context: {RESEARCH_CONTEXT}"""

THEME_REVIEWER = """You review a thematic map (Braun & Clarke, phase 4). Check each theme against the codes:
supported, coherent, distinct? Are the role flags correct? Merge, split, rename or drop themes as needed and
return the full corrected map. Also note concepts that need further consideration."""

def code_transcript(t):
    r = run_agent(CODER, f"Role: {t['user_group']}\n\n{t['clean_text']}", CodingResult)
    print(f"  Coder finished {t['participant_id']}")
    return r

print("Group 2: coders are working...")
for t, c in zip(transcripts, parallel(code_transcript, transcripts)):
    t["codes"] = c.codes
all_codes = "\n\n".join(f"## {t['participant_id']} ({t['user_group']})\n" +
                         "\n".join(f"- {c.code}: {c.description} | \"{c.quote}\"" for c in t["codes"])
                         for t in transcripts)
print("Theme lead is building themes...")
draft = run_agent(THEME_LEAD, f"Codes:\n{all_codes}", ThematicMap, effort="high")
print("Theme reviewer is checking...")
thematic_map = run_agent(THEME_REVIEWER, f"Codes:\n{all_codes}\n\nDraft map:\n{draft.model_dump_json(indent=1)}",
                         ThematicMap, effort="high")
for th in thematic_map.themes:
    print(f"\n■ {th.theme}  [{th.group_flag}]  roles: {', '.join(th.groups_present)}")
    print(f"  {th.definition}")
    if th.group_differences.strip().lower() not in ("", "none"):
        print(f"  Role differences: {th.group_differences}")
theme_brief = "\n".join(f"- {th.theme} [{th.group_flag}]: {th.definition}" for th in thematic_map.themes)
save_json("2_thematic_analysis.json", {"codes": {t["participant_id"]: [c.model_dump() for c in t["codes"]]
                                                 for t in transcripts}, "thematic_map": thematic_map.model_dump()})

## Step 8: Agent group 3, extract statements and rate them 1–5

For each participant:
1. The **Extractor** copies every participant statement that fits a construct definition. A statement can fit more than one construct.
2. **Rater A** and **Rater B** each rate every statement 1–5 on its construct, working independently. They judge both the qualitative meaning and the sentiment of the statement.
3. Where the raters differ by 2 or more points, the **Adjudicator** decides. Otherwise the two ratings are averaged.
4. Statement ratings are averaged into a **participant-level score per attribute**, as in the paper.

In [ ]:
import pandas as pd

EXTRACTOR = f"""You extract evidence for predefined adoption constructs from a clinician's session.
Copy, VERBATIM, every statement by the PARTICIPANT (never the interviewer) that fits one of these
construct definitions. A statement may fit more than one construct; list it once per construct.
Use only the definitions below; do not invent new constructs. List constructs with no evidence.

{attribute_brief()}

Research context: {RESEARCH_CONTEXT}"""

def rater_prompt(name):
    return f"""You are {name}, an independent rater in a qualitative study of technology adoption.
Rate each statement on its assigned construct from 1 (low level of the construct) to 5 (high level),
using only the definitions and anchors below. Consider both what the clinician says and the sentiment
of how they say it (enthusiastic, hesitant, conditional, negative). 3 = neutral, mixed or conditional.
Rate each statement on its own merits.

{attribute_brief()}"""

ADJUDICATOR = f"""You are the senior adjudicator. Two independent raters disagreed by 2+ points on these
statements. Read the statement, its construct definition and both rationales, then give the final 1-5
rating with a rationale.

{attribute_brief()}"""

def verbatim(quote, text):
    norm = lambda s: re.sub(r"[^a-z0-9 ]", "", re.sub(r"\s+", " ", s.lower())).strip()
    q = norm(quote.strip('"“”'))
    return bool(q) and q in norm(text)

def rate_participant(t):
    ext = run_agent(EXTRACTOR, f"Participant {t['participant_id']} ({t['user_group']})\n\n{t['clean_text']}",
                    Extraction, effort="high")
    stmts = [s for s in ext.statements if s.attribute in ATTRIBUTES]
    if not stmts:
        return []
    sheet = "\n".join(f"[{i}] construct={s.attribute}: \"{s.statement}\"" for i, s in enumerate(stmts))
    task = f"Participant {t['participant_id']} ({t['user_group']}). Statements to rate:\n{sheet}"
    a, b = (run_agent(rater_prompt(n), task, RatingSheet) for n in ("Rater A", "Rater B"))
    ra = {r.statement_id: r for r in a.ratings}
    rb = {r.statement_id: r for r in b.ratings}

    rows = []
    for i, s in enumerate(stmts):
        x = min(max(ra[i].rating, 1), 5) if i in ra else None
        y = min(max(rb[i].rating, 1), 5) if i in rb else None
        rows.append({"participant_id": t["participant_id"], "user_group": t["user_group"], "statement_id": i,
                     "attribute": s.attribute, "statement": s.statement,
                     "verbatim": verbatim(s.statement, t["raw_text"]),
                     "rater_a": x, "rater_b": y,
                     "rationale_a": ra[i].rationale if i in ra else "", "rationale_b": rb[i].rationale if i in rb else ""})

    disputed = [r for r in rows if None not in (r["rater_a"], r["rater_b"]) and abs(r["rater_a"] - r["rater_b"]) >= 2]
    final = {}
    if disputed:
        adj = run_agent(ADJUDICATOR, "\n\n".join(
            f"[{r['statement_id']}] construct={r['attribute']}: \"{r['statement']}\"\n"
            f"Rater A={r['rater_a']} ({r['rationale_a']})\nRater B={r['rater_b']} ({r['rationale_b']})"
            for r in disputed), RatingSheet)
        final = {r.statement_id: min(max(r.rating, 1), 5) for r in adj.ratings}
    for r in rows:
        given = [v for v in (r["rater_a"], r["rater_b"]) if v is not None]
        r["adjudicated"] = r["statement_id"] in final
        r["rating"] = final.get(r["statement_id"], sum(given) / len(given) if given else None)
    print(f"  {t['participant_id']}: {len(rows)} statements, {len(disputed)} adjudicated")
    return rows

print("Group 3: extractor, raters and adjudicator are working...")
statements = pd.DataFrame([r for rows in parallel(rate_participant, transcripts) for r in rows])
statements = statements[statements["verbatim"]].reset_index(drop=True)   # keep only quotes found in the transcript

# Inter-rater agreement
from sklearn.metrics import cohen_kappa_score
both = statements.dropna(subset=["rater_a", "rater_b"])
kappa = cohen_kappa_score(both["rater_a"].astype(int), both["rater_b"].astype(int), weights="quadratic") \
        if both["rater_a"].nunique() > 1 or both["rater_b"].nunique() > 1 else float("nan")
print(f"\nStatements kept: {len(statements)}  |  exact agreement {(both.rater_a == both.rater_b).mean():.0%}"
      f"  |  within 1 point {((both.rater_a - both.rater_b).abs() <= 1).mean():.0%}  |  quadratic weighted kappa {kappa:.2f}")

ratings = (statements.pivot_table(index="participant_id", columns="attribute", values="rating", aggfunc="mean")
           .reindex(index=list(by_pid), columns=ATTRIBUTE_KEYS))
ratings.insert(0, "user_group", [by_pid[p]["user_group"] for p in ratings.index])
statements.to_csv(os.path.join(OUTPUT_DIR, "3_rated_statements.csv"), index=False)
ratings.to_csv(os.path.join(OUTPUT_DIR, "3_participant_ratings.csv"))

from IPython.display import display, Markdown
summary = ratings.groupby("user_group")[ATTRIBUTE_KEYS].agg(["mean", "std"]).round(2)
display(Markdown("**Attribute ratings by role (mean, SD)**"))
display(summary.rename(columns=SHORT))
display(ratings.rename(columns=SHORT).style.format(precision=2, na_rep="–")
        .background_gradient(cmap="Blues", vmin=1, vmax=5, subset=list(SHORT.values())))

### Optional: RoBERTa sentiment scores (as in the paper)

The CardiffNLP RoBERTa model scores every extracted statement. Scores are converted to a 1–5 scale (`3 + 2 × (P(positive) − P(negative))`), averaged per attribute, and kept as **separate clustering features**.

In [ ]:
sentiment = None
if USE_ROBERTA_SENTIMENT:
    from transformers import pipeline
    clf = pipeline("text-classification", model="cardiffnlp/twitter-roberta-base-sentiment-latest",
                   top_k=None, truncation=True, max_length=512)
    def to_1_5(scores):
        p = {s["label"].lower(): s["score"] for s in scores}
        return 3 + 2 * (p.get("positive", 0) - p.get("negative", 0))
    statements["sentiment"] = [to_1_5(s) for s in clf(statements["statement"].tolist(), batch_size=16)]
    sentiment = (statements.pivot_table(index="participant_id", columns="attribute", values="sentiment", aggfunc="mean")
                 .reindex(index=ratings.index, columns=ATTRIBUTE_KEYS))
    statements.to_csv(os.path.join(OUTPUT_DIR, "3_rated_statements.csv"), index=False)
    sentiment.to_csv(os.path.join(OUTPUT_DIR, "3_participant_sentiment.csv"))
    display(Markdown("**Sentiment scores by role (mean)**"))
    display(sentiment.join(ratings["user_group"]).groupby("user_group").mean().round(2).rename(columns=SHORT))

## Step 9: Agent group 4, hierarchical clustering by role

As in the paper, each role is clustered separately: features are z-scored, distances are Euclidean, and the linkage is complete. For each candidate cut point, the **Cluster Analyst** checks whether every cluster's members share coherent adoption perceptions, using their rated statements.

> **Researcher checkpoint:** review the dendrograms and the analyst's recommendation. To overrule it, set `FINAL_K` in the next cell, then run it.

In [ ]:
import numpy as np
import matplotlib.pyplot as plt
from scipy.cluster.hierarchy import linkage, dendrogram, fcluster
from scipy.stats import zscore

MAX_K = 5
features = ratings[ATTRIBUTE_KEYS].add_prefix("rating:")
if sentiment is not None:
    features = features.join(sentiment.add_prefix("sentiment:"))

CLUSTER_ANALYST = f"""You are a qualitative researcher reviewing hierarchical clustering of clinicians'
adoption attribute scores (one professional role at a time). Clusters are an organizing step, not inference.
For each candidate number of clusters, judge whether the members of each cluster share coherent adoption
perceptions according to their rated statements. Consider dendrogram structure, linkage distances, cluster
sizes and qualitative coherence. Recommend the number of clusters. A single-member cluster is acceptable when
that participant's adoption pattern is clearly distinct. Research context: {RESEARCH_CONTEXT}"""

def statements_of(pid, limit=12):
    s = statements[statements["participant_id"] == pid]
    return "\n".join(f"  - {SHORT[r.attribute]} ({r.rating:.1f}): \"{r.statement}\"" for r in s.head(limit).itertuples())

cluster_data = {}
fig, axes = plt.subplots(1, len(USER_GROUPS), figsize=(7 * len(USER_GROUPS), 4.5), squeeze=False)
for ax, group in zip(axes[0], USER_GROUPS):
    F = features[ratings["user_group"] == group]
    if len(F) < 3:
        print(f"{group}: only {len(F)} participants, clustering skipped (one cluster).")
        cluster_data[group] = {"F": F, "Z": None, "candidates": {1: np.ones(len(F), int)}, "recommended": 1}
        ax.set_visible(False)
        continue
    X = F.apply(lambda c: c.fillna(c.median())).fillna(3)          # missing evidence -> role median
    Xz = X.apply(lambda c: zscore(c) if c.std() > 0 else c * 0)
    Z = linkage(Xz.values, method="complete", metric="euclidean")
    candidates = {k: fcluster(Z, k, criterion="maxclust") for k in range(2, min(MAX_K, len(F) - 1) + 1)}
    dendrogram(Z, labels=list(F.index), ax=ax, color_threshold=0, above_threshold_color="#52514e")
    ax.set_title(f"{group}s: complete linkage, Euclidean, z-scored (n={len(F)})", loc="left", fontsize=11)
    ax.set_ylabel("Linkage distance"); ax.spines[["top", "right"]].set_visible(False)

    desc = []
    for k, lab in candidates.items():
        desc.append(f"### k = {k} (cut height range: {Z[-k, 2]:.2f}-{Z[-(k - 1), 2]:.2f})")
        for c in sorted(set(lab)):
            mem = list(F.index[lab == c])
            means = ratings.loc[mem, ATTRIBUTE_KEYS].mean().round(2).rename(SHORT).to_dict()
            desc.append(f"Cluster {c} (n={len(mem)}): {mem}\nMean ratings: {means}")
    evidence = "\n".join(f"{p}:\n{statements_of(p)}" for p in F.index)
    rec = run_agent(CLUSTER_ANALYST, f"Role: {group}\nMerge heights: {np.round(Z[:, 2], 2).tolist()}\n\n"
                    + "\n\n".join(desc) + f"\n\nRated statements per participant:\n{evidence}",
                    CutRecommendation, effort="high")
    rk = rec.recommended_k if rec.recommended_k in candidates else min(candidates)
    cluster_data[group] = {"F": F, "Z": Z, "candidates": candidates, "recommended": rk, "recommendation": rec}
    print(f"\n{group}s, analyst recommends k = {rk}\n  {rec.rationale}")
    for line in rec.coherence_by_k:
        print("   •", line)
plt.tight_layout(); plt.savefig(os.path.join(OUTPUT_DIR, "4_dendrograms.png"), dpi=150); plt.show()

In [ ]:
# ---------------- RESEARCHER CHECKPOINT ----------------
FINAL_K = {"Physician": None, "Nurse": None}    # None = accept the analyst's recommendation
# --------------------------------------------------------

cluster_of = {}
for group, d in cluster_data.items():
    k = FINAL_K.get(group) or d["recommended"]
    labels = d["candidates"][k]
    for pid, c in zip(d["F"].index, labels):
        cluster_of[pid] = f"{group}-{c}"
ratings["cluster"] = ratings.index.map(cluster_of)
clusters = sorted(set(cluster_of.values()))
for c in clusters:
    print(f"{c}: {list(ratings.index[ratings['cluster'] == c])}")
display(ratings.groupby("cluster")[ATTRIBUTE_KEYS].mean().round(2).rename(columns=SHORT)
        .style.format(precision=2).background_gradient(cmap="Blues", vmin=1, vmax=5))

PROFILE_MATCHER = f"""You compare clusters from separate physician and nurse analyses. Clusters from different
roles with similar adoption patterns (attribute ratings AND underlying statements: perceived clinical value,
concerns, conditions for adoption) represent the SAME adoption profile; merge them. Keep workflow and
responsibility differences as role differences within a profile. Every cluster must appear in exactly one
profile. Give each profile a short, design-relevant name. Research context: {RESEARCH_CONTEXT}"""

print("\nProfile matcher is comparing physician and nurse clusters...")
matching = run_agent(PROFILE_MATCHER, "\n\n".join(
    f"## {c} (n={sum(ratings['cluster'] == c)})\nMean ratings: "
    f"{ratings.loc[ratings['cluster'] == c, ATTRIBUTE_KEYS].mean().round(2).rename(SHORT).to_dict()}\n"
    + "\n".join(f"{p}:\n{statements_of(p, 8)}" for p in ratings.index[ratings["cluster"] == c])
    for c in clusters) + f"\n\nThemes:\n{theme_brief}", ProfileMatching, effort="high")

profile_of = {}
for prof in matching.profiles:
    for c in prof.clusters:
        if c in clusters and c not in profile_of:
            profile_of[c] = prof.profile_name
for c in clusters:                                        # any cluster the matcher missed keeps its own profile
    profile_of.setdefault(c, f"Profile {c}")
ratings["profile"] = ratings["cluster"].map(profile_of)
profiles = list(dict.fromkeys(ratings["profile"]))
print("\n", matching.rationale)
print(pd.crosstab(ratings["profile"], ratings["user_group"], margins=True))
save_json("4_clusters_and_profiles.json", {
    "final_k": {g: FINAL_K.get(g) or d["recommended"] for g, d in cluster_data.items()},
    "recommendations": {g: d["recommendation"].model_dump() for g, d in cluster_data.items() if "recommendation" in d},
    "cluster_of": cluster_of, "matching": matching.model_dump()})
ratings.to_csv(os.path.join(OUTPUT_DIR, "4_participants_with_profiles.csv"))

## Step 10: Agent group 5, persona cards

In [ ]:
import html as _html

PERSONA_DESIGNER = f"""You design evidence-based CLINICIAN ADOPTION PERSONAS. Write one persona for the given
adoption profile, grounded in its members' rated statements, attribute ratings and the themes. Quotes must
be copied verbatim from participants (never the interviewer). Describe how this profile evaluates the system,
what drives and blocks adoption, and the conditions for use. Give role-specific needs for each role present.
Design requirements must be concrete (dashboard content, presentation, explanations). Implementation
strategies must match the profile's readiness (training, evidence, trial use, peer support).
Research context: {RESEARCH_CONTEXT}"""

prof_info = {p.profile_name: p for p in matching.profiles}

def design_persona(name):
    mem = list(ratings.index[ratings["profile"] == name])
    p = prof_info.get(name)
    member_list = ", ".join(f"{m} ({by_pid[m]['user_group']})" for m in mem)
    task = (f"Adoption profile: {name}\nPattern: {p.adoption_pattern if p else ''}\n"
            f"Role differences: {p.role_differences if p else ''}\n"
            f"Members: {member_list}\n"
            f"Mean ratings: {ratings.loc[mem, ATTRIBUTE_KEYS].mean().round(2).rename(SHORT).to_dict()}\n\n"
            f"Themes:\n{theme_brief}\n\nMember transcripts:\n\n"
            + "\n\n".join(f"### {m} ({by_pid[m]['user_group']})\n{by_pid[m]['clean_text']}" for m in mem))
    card = run_agent(PERSONA_DESIGNER, task, PersonaCard, effort="high")
    print(f"  Persona designer finished: {name}")
    return card

print("Group 5: persona designers are working...")
cards = dict(zip(profiles, parallel(design_persona, profiles)))

SERIES = ["#2a78d6", "#eb6834", "#1baf7a", "#eda100", "#e87ba4", "#008300", "#6250d6", "#e34948"]
def _li(items): return "".join(f"<li>{_html.escape(i)}</li>" for i in items)
def _q(quote, mem):
    ok = any(verbatim(quote, by_pid[m]["raw_text"]) for m in mem)
    return (f'<blockquote>“{_html.escape(quote.strip(chr(34) + "“”"))}”'
            f'<span class="chk">{"✓ verified in transcript" if ok else "⚠ not found in transcripts"}</span></blockquote>')

def card_html(i, name, card):
    mem = list(ratings.index[ratings["profile"] == name])
    color = SERIES[i % len(SERIES)]
    comp = ", ".join(f"{n} {g.lower()}{'s' if n > 1 else ''}"
                     for g, n in ratings.loc[mem, "user_group"].value_counts().items())
    bars = ""
    for k in ATTRIBUTE_KEYS:
        v = ratings.loc[mem, k].mean()
        width, label = (0, "–") if pd.isna(v) else (v / 5 * 100, f"{v:.2f}")
        bars += (f'<div class="bar"><span>{_html.escape(SHORT[k])}</span><div class="track">'
                 f'<div class="fill" style="width:{width:.0f}%"></div></div><b>{label}</b></div>')
    roles = "".join(f"<div><h4>{_html.escape(r.role)}</h4><ul>{_li(r.needs)}</ul></div>" for r in card.role_specific_needs)
    words = [w for w in card.persona_name.split() if w.rstrip(".").lower() not in ("dr", "nurse", "rn", "md", "np")]
    initials = "".join(w[0] for w in words[:2]).upper()
    return f"""
<div class="card">
  <div class="head" style="border-top:6px solid {color}">
    <div class="avatar" style="background:{color}">{_html.escape(initials)}</div>
    <div><div class="profile">{_html.escape(name)}</div><h2>{_html.escape(card.persona_name)}</h2>
    <span class="meta">{len(mem)} of {len(ratings)} clinicians · {comp}</span></div>
  </div>
  {_q(card.tagline_quote, mem)}
  <p>{_html.escape(card.description)}</p>
  <h3>Adoption attributes <small>(mean qualitative rating, 1–5)</small></h3>{bars}
  <div class="cols">
    <div><h3>Goals</h3><ul>{_li(card.goals)}</ul></div>
    <div><h3>Conditions for use</h3><ul>{_li(card.conditions_for_use)}</ul></div>
    <div><h3>Adoption drivers</h3><ul>{_li(card.adoption_drivers)}</ul></div>
    <div><h3>Adoption concerns</h3><ul>{_li(card.adoption_concerns)}</ul></div>
  </div>
  <h3>Role-specific needs</h3><div class="cols">{roles}</div>
  <h3>Design requirements</h3><ul>{_li(card.design_requirements)}</ul>
  <h3>Implementation strategies</h3><ul>{_li(card.implementation_strategies)}</ul>
  <h3>In their words</h3>{"".join(_q(q, mem) for q in card.supporting_quotes)}
</div>"""

CSS = """
:root{--bg:#f4f4f2;--card:#fcfcfb;--ink:#0b0b0b;--ink2:#52514e;--line:#e4e3df;--track:#e9e8e4;--fill:#2a78d6}
@media (prefers-color-scheme: dark){:root{--bg:#111110;--card:#1a1a19;--ink:#f0efec;--ink2:#c3c2b7;--line:#383835;--track:#2c2c2a;--fill:#3987e5}}
body{background:var(--bg);margin:0}
.wrap{font-family:system-ui,-apple-system,Segoe UI,Roboto,sans-serif;color:var(--ink);display:grid;
      grid-template-columns:repeat(auto-fill,minmax(380px,1fr));gap:20px;padding:16px;background:var(--bg)}
.card{background:var(--card);border:1px solid var(--line);border-radius:14px;padding:20px;font-size:14px;line-height:1.45}
.head{display:flex;gap:14px;align-items:center;margin:-20px -20px 8px;padding:16px 20px 0;border-radius:14px 14px 0 0}
.avatar{width:56px;height:56px;border-radius:50%;color:#fff;display:flex;align-items:center;justify-content:center;font-weight:700;font-size:20px;flex:none}
.profile{font-size:12px;font-weight:700;text-transform:uppercase;letter-spacing:.06em;color:var(--ink2)}
h2{margin:0;font-size:20px} h3{font-size:12px;text-transform:uppercase;letter-spacing:.06em;color:var(--ink2);margin:16px 0 6px}
h3 small{text-transform:none;letter-spacing:0;font-weight:400} h4{margin:4px 0;font-size:13px}
.meta{display:block;font-size:12px;color:var(--ink2);margin-top:4px}
blockquote{margin:10px 0;padding:8px 12px;border-left:3px solid var(--line);font-style:italic}
.chk{display:block;font-style:normal;font-size:11px;color:var(--ink2);margin-top:2px}
.bar{display:grid;grid-template-columns:160px 1fr 36px;gap:8px;align-items:center;font-size:12px;color:var(--ink2);margin:3px 0}
.track{height:8px;background:var(--track);border-radius:4px}.fill{height:8px;background:var(--fill);border-radius:4px}
.bar b{color:var(--ink);text-align:right}
.cols{display:grid;grid-template-columns:1fr 1fr;gap:0 16px} ul{margin:0;padding-left:18px}
@media (max-width:480px){.cols{grid-template-columns:1fr}.wrap{grid-template-columns:1fr}}
"""
cards_html = f'<style>{CSS}</style><div class="wrap">' + "".join(
    card_html(i, n, cards[n]) for i, n in enumerate(profiles)) + "</div>"
from IPython.display import HTML
display(HTML(cards_html))
with open(os.path.join(OUTPUT_DIR, "5_persona_cards.html"), "w", encoding="utf-8") as f:
    f.write('<!doctype html><html><head><meta charset="utf-8"><meta name="viewport" content="width=device-width,'
            f'initial-scale=1"><title>Clinician Adoption Personas</title></head><body>{cards_html}</body></html>')
save_json("5_persona_cards.json", {n: c.model_dump() for n, c in cards.items()})

## Step 11 (optional): Post hoc correlations with usability measures

Upload a CSV with columns `participant_id`, `SUS`, `PEOU` (for example, the SUS score and perceived ease of use on a 1–7 scale). The notebook computes Pearson correlations with the attribute ratings, as in the paper's post hoc analysis.

In [ ]:
RUN_POSTHOC = False

if RUN_POSTHOC:
    from scipy.stats import pearsonr
    from matplotlib.colors import LinearSegmentedColormap
    from google.colab import files
    import io
    up = files.upload()
    usab = pd.read_csv(io.BytesIO(next(iter(up.values())))).set_index("participant_id")
    data = ratings[ATTRIBUTE_KEYS].rename(columns=SHORT).join(usab[["SUS", "PEOU"]], how="inner")
    cols = list(data.columns)
    r = data.corr(method="pearson")
    p = pd.DataFrame([[pearsonr(*data[[a, b]].dropna().T.values)[1] if a != b else 0 for b in cols] for a in cols],
                     index=cols, columns=cols)
    diverging = LinearSegmentedColormap.from_list("div", ["#e34948", "#f0efec", "#2a78d6"])
    fig, ax = plt.subplots(figsize=(8, 6.5))
    im = ax.imshow(r.values, cmap=diverging, vmin=-1, vmax=1)
    ax.set_xticks(range(len(cols)), cols, rotation=40, ha="right"); ax.set_yticks(range(len(cols)), cols)
    for i in range(len(cols)):
        for j in range(len(cols)):
            star = "*" if i != j and p.iloc[i, j] < .05 else ""
            ax.text(j, i, f"{r.iloc[i, j]:.2f}{star}", ha="center", va="center", fontsize=9, color="#0b0b0b")
    fig.colorbar(im, ax=ax, label="Pearson r"); ax.set_title(f"Correlations (N={len(data)}; * P<.05)", loc="left")
    plt.tight_layout(); plt.savefig(os.path.join(OUTPUT_DIR, "6_posthoc_correlations.png"), dpi=150); plt.show()
    r.round(2).to_csv(os.path.join(OUTPUT_DIR, "6_posthoc_correlations.csv"))

## Step 12: Download everything

In [ ]:
import shutil
shutil.make_archive("persona_outputs", "zip", OUTPUT_DIR)
print("Files:", sorted(os.listdir(OUTPUT_DIR)))
try:
    from google.colab import files
    files.download("persona_outputs.zip")
except ImportError:
    print("Saved persona_outputs.zip")